In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red">ch2. LLM활용의 기본 개념(Ollama)</span>
  
# 1. LLM을 활용하여 답변 생성하기
    
## 1) Ollama이용한 로컬 LLM이용
성능은 GPT, Claude같은 모델보다 떨어지나, 개념 설명을 위해 opensource모델 사용

### (a) ollama.com 다운로드 -> 설치 -> 모델pull
- ollama run deepseek-r1:1.5b (window키+R => powershell)

In [3]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="deepseek-r1:1.5b")
result = llm.invoke("What is the capital of Korea?")
result

AIMessage(content='<think>\n\n</think>\n\nThe capital of Korea, which is South Korea, is Seoul.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-26T01:49:59.2307332Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3711196000, 'load_duration': 2506189100, 'prompt_eval_count': 10, 'prompt_eval_duration': 262845700, 'eval_count': 18, 'eval_duration': 939447700, 'model_name': 'deepseek-r1:1.5b'}, id='run--0f72e053-4f89-4143-9cf0-73d0c110491b-0', usage_metadata={'input_tokens': 10, 'output_tokens': 18, 'total_tokens': 28})

### ollama.com다운로드 -> 설치 -> 모델 pull
- ollama pull llama3.2:1b (window키+R => powershell)
- llama : 공식적으로 한글지원 안 됨(llama3.1 4.5b한글지원 가능 -> llama3.3 70b)
- exaone : 공식적으로 한글지원

In [4]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")
result = llm.invoke("What is the capital of Korea?")
result.content

"The capital of South Korea is Seoul. However, it's worth noting that the country also has a special administrative region called Gangwon-do and an autonomous city-state called Jeju Island, but the two are not officially recognized as separate countries or regions."

In [5]:
result = llm.invoke("한국 수도는 어디야?")
result.content

' 한국 수도는 전주시입니다.'

## 2. openai 활용
- pip install langchain-openai

In [8]:
from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")
# result = llm.invoke("What is the capital of Korea?")
# result => 에러 이유 : OPENAI_API_KEY 환경변수 부재

In [12]:
#환경 변수 가져오기 
from dotenv import load_dotenv
import os
load_dotenv()
#os.getenv('OPENAI_API_KEY')

True

In [13]:
#코랩에서 OPENAI_API_KEY읽어오기(.env못씀)
#보안키 추가 후
# from google.colab import userdata
# userdata.get('OPENAI_API_KEY')

In [14]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano", 
                 #openai_api_key=os.getenv('OPEN_API_KEY')
                )
llm.invoke("What is the capital of Korea? Answer me in Korean")

AIMessage(content='대한민국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 18, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_f12167b370', 'id': 'chatcmpl-BmBeZ4SEDYKFfAjPenW8VTKzMrns8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--e66ca837-7ecb-46f7-91bb-41f951d08d8c-0', usage_metadata={'input_tokens': 18, 'output_tokens': 8, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
# 모든 모델의 키가 OPENAI_API_KEY
# Claude -> Anthropic
# Azure upstage, Bedrock : 에러 메세지 참조하여 환경변수 생성

In [17]:
# from langchain_openai import AzureOpenAI
# llm = AzureOpenAI(model="gpt-4.1-nano")
# 에러를 내면 OPENAI_API_VERSION 환경변수가 필요하는 메세지

In [19]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-3-5-sonnet-20240620")
#llm.invoke("What is the capital of Korea?")
#에러 메세지를 봐도 환경변수

# 2. 렝체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질문

In [21]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")
#llm.invoke(0)
#프로프트 타입 : 스트링, PromptValue, BaseMessage리스트

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplete을 사용하여 변수가 포함된 템플릿 작성

In [25]:
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model="llama3.2:1b")
prompt_template = PromptTemplate(
    template="What is the capital of {country}", #{}안의 값을 새로운 값으로 대입 가능
    input_variables = ["country"]
)
llm.invoke(prompt_template.invoke({"country":"Korea"}))
print(prompt)
llm.invoke(prompt)

AIMessage(content="The capital of South Korea is Seoul. The government and most major institutions are located in Seoul, although some areas like Gyeonggi Province and Gangwon-do Province are also home to a significant portion of the country's population and economic activity.", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T05:31:31.6292191Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3227600300, 'load_duration': 18628300, 'prompt_eval_count': 31, 'prompt_eval_duration': 58965400, 'eval_count': 50, 'eval_duration': 3149493600, 'model_name': 'llama3.2:1b'}, id='run--b406b197-8c6f-46d2-99da-43159a5ce762-0', usage_metadata={'input_tokens': 31, 'output_tokens': 50, 'total_tokens': 81})

## 2) 메세지 기반 프롬프트 작성
- BaseMessage리스트
- BaseMessage상속 받은 클래스 : AIMessage, HummanMessage, SystemMessage, ToolMessage

In [34]:
# BaseMEssage list로 하면 렝체인화x, ChatPromptTemplate X
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
llm = ChatOllama(model="llama3.2:1b")
message_list = [
    SystemMessage(content="You are a helpful assistant!"),
    HumanMessage(content="What is the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content="What is the capital of Korea?"),
    AIMessage(content="The capital of Korea is Seoul."),
    HumanMessage(content="What is the capital of {country}?")
]
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate.from_messages(message_list)
prompt = chatPromptTemplate.invoke({"country":"Korea"})
print(prompt)

messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of {country}?', additional_kwargs={}, response_metadata={})]


## 3) ChatPromptTemplate사용


In [38]:
#위의 BaseMessage를 수정
chatPromptTemplate = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant!"),
    ("human", "What is the capital of {country}?"),
])
country = input("어느 나라 수도가 궁금하세요?")
prompt = chatPromptTemplate.invoke({"country":country})
print(prompt)
result = llm.invoke(prompt)
result.content

어느 나라 수도가 궁금하세요?한국
messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of 한국?', additional_kwargs={}, response_metadata={})]


'The capital of South Korea is Seoul. Is there anything else I can help you with regarding Korean culture or history?'

# 3. 답변 형식을 컨트롤하기
- invoke 시행결과는 AIMessage() -> String이나 json, 객체 : outputParser이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 사용하여 LLM출력(AIMessage)을 단순 문자열로 변환

In [44]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template="what is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({"country":"Korea"})
print("프롬프트 :", prompt)
result = llm.invoke(prompt)
print('llm결과 :', type(result), result)
#문자열 출력 파서를 이용하여 llm응답을 단순 문자열 변환
output_parser = StrOutputParser()
print('파서 결과 :', output_parser.invoke(result))

프롬프트 : text='what is the capital of Korea. Return the name of the city only'
llm결과 : <class 'langchain_core.messages.ai.AIMessage'> content='Seoul' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T06:39:46.8592963Z', 'done': True, 'done_reason': 'stop', 'total_duration': 221992500, 'load_duration': 28204700, 'prompt_eval_count': 39, 'prompt_eval_duration': 73137400, 'eval_count': 3, 'eval_duration': 120627200, 'model_name': 'llama3.2:1b'} id='run--5386a9f7-30d5-4a57-8996-e0072ed970a3-0' usage_metadata={'input_tokens': 39, 'output_tokens': 3, 'total_tokens': 42}
파서 결과 : Seoul


In [50]:
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Korea"})))

'Seoul'

In [51]:
# PromptTemplate(변수설정) => ChatPromptTemplate(변수설정, system과 모범답안 지정)
llm = ChatOllama(model="llama3.2:1b")

chat_prompt_template = ChatPromptTemplate([
    ("system", "You are a helful assistant with expertise in South Korea"),
    ("human", "What is the capital of {country}? Return the name if the city only")
])

output_parser = StrOutputParser()

output_parser.invoke(llm.invoke(chat_prompt_template.invoke({"country":"Korea"})))

'Seoul'

## 2) JSON출력 파서 이용
- json()으로 응답하기를 원하지만, 우선 어떤 형식으로 반환되는지 확인
- {"name":"홍", "age":22}(json) / {'name':'홍', 'age':22}(dict)

In [56]:
from langchain_core.output_parsers import JsonOutputParser
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    -Capital
    -Population
    -Language
    -Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
prompt = country_detail_prompt.invoke({"country":"Korea"})
print(type(prompt))
#Json output 파서
output_parser = JsonOutputParser()
ai_message = llm.invoke(prompt)
print(type(prompt), ai_message)
json_result = output_parser.invoke(ai_message)
print(json_result)

<class 'langchain_core.prompt_values.StringPromptValue'>
<class 'langchain_core.prompt_values.StringPromptValue'> content='```json\n{\n    "capital": "Seoul",\n    "population": 51.8,\n    "language": "Korean",\n    "currency": "KRW"\n}\n```' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T07:01:31.3649735Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2520759700, 'load_duration': 26782000, 'prompt_eval_count': 59, 'prompt_eval_duration': 64939500, 'eval_count': 40, 'eval_duration': 2428548300, 'model_name': 'llama3.2:1b'} id='run--b339e262-c957-4ed0-aa75-2f8d7f6280f5-0' usage_metadata={'input_tokens': 59, 'output_tokens': 40, 'total_tokens': 99}
{'capital': 'Seoul', 'population': 51.8, 'language': 'Korean', 'currency': 'KRW'}


In [61]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    -Capital
    -Population
    -Language
    -Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
output_parser = JsonOutputParser()
info = output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))
info

{'capital': 'Seoul',
 'population': '51.8 million (2021 estimate)',
 'language': 'Korean (official), Korean Hanja, English',
 'currency': 'South Korean won'}

In [62]:
type(info)

dict

## 3) 구조화된 출력 사용
- Pydantic 모델을 사용하여 LLM 출력을 구조화된 형식으로 받기(JsonParser보다 훨씬 안정적)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리

In [66]:
from pydantic import BaseModel, Field
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
#     def __str__(self):
#         return self.name + " " + self.id
user = User(1, "홍길동")
print(user)

In [65]:
class User(BaseModel):
    #gt=0:id>0 , ge=0:id>=0 , lt=0:id<0, le=0:id<=0
    id : int = Field(gt=0,            description="id")
    name:str = Field(min_length=2,    description="name")
    is_active:bool=Field(default=True,description="id활성화")
user = User(id=1, name="홍길동")
print(user)

id=1 name='홍길동' is_active=True


In [72]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    -Capital
    -Population
    -Language
    -Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
class CountryDetail(BaseModel): #description : 더 정확한 출력 유도
    capital:str    = Field(description="the capital of the country")
    population:int = Field(description="the population of the country")
    language:str   = Field(description="the language of the country")
    currency:str   = Field(description="the currency of the country")
# 출력 형식 파서 + LLM
structedllm = llm.with_structured_output(CountryDetail)

# output_parser = JsonOutputParser()
# output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))

info = structedllm.invoke(country_detail_prompt.invoke({"country":"Korea"}))
type(info)

__main__.CountryDetail

In [73]:
print(info)
print(info.capital, info.population)

capital='Seoul' population=51 language='Korean (official)' currency='South Korean won'
Seoul 51


In [74]:
print('info를 json :', info.model_dump_json()) #json()
print('info를 dict :', info.dict())

info를 json : {"capital":"Seoul","population":51,"language":"Korean (official)","currency":"South Korean won"}
info를 dict : {'capital': 'Seoul', 'population': 51, 'language': 'Korean (official)', 'currency': 'South Korean won'}


C:\Users\Admin\AppData\Local\Temp\ipykernel_13556\520016047.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print('info를 dict :', info.dict())


# 4. LCEL을 활용한 렝체인 생성하기
## 1) 문자열 출력 파서 사용
- invoke

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model="llama3.2:1b", 
                temperature=0) #일관된 답변
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template="what is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Korea"})))

'Seoul'

## 2) LCEL을 사용한 간단한 체인 구성
- 파이프연산자(|)사용

In [5]:
# 프롬프트템플릿 -> llm -> 출력파서를 연결하는 체인 생성
capital_chain = prompt_template | llm | output_parser
#생성된 체인 invoke
capital_chain.invoke({"country":"Korea"})

'Seoul'

In [6]:
type(capital_chain)

langchain_core.runnables.base.RunnableSequence

## 3) 복합 체인 구성
- 여러단계의 추론이 필요한 경우 (체인 연결)

In [7]:
# 나라 설명 입력 -> 나라명 출력
from langchain_core.prompts import PromptTemplate
country_prompt = PromptTemplate(
    template = """Guess the name of the country based on the following information:
    {information}
    return the name of the country only""",
    input_variables=["information"]
)
output_parser.invoke(llm.invoke(country_prompt.invoke({"information":
                                                       "This country is very famous for its wine"})))

'Italy.'

In [9]:
# 나라명 추측 체인 생성
country_chain = country_prompt | llm | output_parser
#type(country_chain)
country_chain.invoke({"information":"This country is very famous for its wine"})

'Italy.'

In [11]:
# 나라설명 -> (나라명 -> ) 그 나라 수도
final_chain = country_chain | capital_chain
final_chain.invoke({"information":"This country is very famous for its wine"})

'Rome'

In [14]:
final_chain = {"country":country_chain} | capital_chain
final_chain.invoke({"information":"This country is very famous for its wine"})

'Rome'

In [15]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {"information":RunnablePassthrough()}|{"country":country_chain}|capital_chain
final_chain.invoke({"This country is very famous for its wine"})

'Paris'

In [17]:
# 프롬프트 템플릿에 변수 2개
# 나라 설명 입력 -> 나라명 출력
from langchain_core.prompts import PromptTemplate
country_prompt = PromptTemplate(
    template = """Guess the name of the country in the {continent} based on the following information:
    {information}
    return the name of the country only""",
    input_variables=["continent","information"]
)
# output_parser.invoke(llm.invoke(country_prompt.invoke({"information":
#                                                        "This country is very famous for its wine",
#                                                      "continent":"Europe"})))
country_chain = country_prompt | llm | output_parser
country_chain.invoke({"information":"This country is very famous for its wine",
                     "continent":"Europe"})

'Italy.'

In [18]:
final_chain = {"country":country_chain} | capital_chain
final_chain.invoke({"information":"This country is very famous for its wine",
                     "continent":"Europe"})

'Rome'